# 02 – Cohort Construction (Angus sepsis definition)

Builds the Angus administrative sepsis cohort from raw MIMIC-III tables.

**Run first** — uses raw MIMIC tables (DIAGNOSES_ICD, PROCEDURES_ICD, ICUSTAYS,
ADMISSIONS, PATIENTS). The Angus sepsis definition (Angus et al., 2001) is applied via
the official SQL from the MIT-LCP mimic-code repository, downloaded in the cell below.

**Produces:** final_cohort_angus.csv (used by notebooks 01, 03, 04, 06).

MIMIC-III data not included (PhysioNet DUA); see README. Patient-row outputs cleared.

In [ ]:
from urllib.request import urlretrieve

url = "https://raw.githubusercontent.com/MIT-LCP/mimic-code/main/mimic-iii/concepts/sepsis/angus.sql"

urlretrieve(url, "angus.sql")

f = open("angus.sql", "r", encoding="utf-8")

print(f.read())

In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import pandas as pd
import numpy as np


## 1. Load tables
Angus needs DIAGNOSES_ICD, PROCEDURES_ICD (for mechanical ventilation), ADMISSIONS, ICUSTAYS, PATIENTS.

In [ ]:
diagnoses  = pd.read_csv(data_path("DIAGNOSES_ICD.csv"))
procedures = pd.read_csv(data_path("PROCEDURES_ICD.csv.gz"))
icustays   = pd.read_csv(data_path("ICUSTAYS.csv.gz"))
admissions = pd.read_csv(data_path("ADMISSIONS.csv.gz"))
patients   = pd.read_csv(data_path("PATIENTS.csv.gz"))

In [ ]:
# ICD9_CODE is stored as string in MIMIC; make sure and strip whitespace
for df in (diagnoses, procedures):
    df["ICD9_CODE"] = df["ICD9_CODE"].astype(str).str.strip()

## 2. Angus infection codes (Appendix 1)
Matched by ICD-9 **prefix** (first 3 / 4 / 5 characters), exactly as in `angus.sql`.

In [ ]:
# --- Infection code prefixes (verbatim from official angus.sql, Appendix 1) ---
infection_3 = ['001','002','003','004','005','008',
    '009','010','011','012','013','014','015','016','017','018',
    '020','021','022','023','024','025','026','027','030','031',
    '032','033','034','035','036','037','038','039','040','041',
    '090','091','092','093','094','095','096','097','098','100',
    '101','102','103','104','110','111','112','114','115','116',
    '117','118','320','322','324','325','420','421','451','461',
    '462','463','464','465','481','482','485','486','494','510',
    '513','540','541','542','566','567','590','597','601','614',
    '615','616','681','682','683','686','730']

infection_4 = ['5695','5720','5721','5750','5990','7110',
    '7907','9966','9985','9993']

infection_5 = ['49121','56201','56203','56211','56213','56983']

def is_infection(code):
    return (code[:3] in infection_3) or (code[:4] in infection_4) or (code[:5] in infection_5)

diagnoses["infection"] = diagnoses["ICD9_CODE"].apply(is_infection).astype(int)

## 3. Angus organ-dysfunction codes + explicit sepsis (Appendix 2)

In [ ]:
# --- Acute organ dysfunction prefixes (verbatim from angus.sql) ---
organ_3 = ['458','293','570','584']
organ_4 = ['7855','3483','3481','2874','2875','2869','2866','5734']

def is_organ_dysfunction(code):
    return (code[:3] in organ_3) or (code[:4] in organ_4)

diagnoses["organ_dysfunction"] = diagnoses["ICD9_CODE"].apply(is_organ_dysfunction).astype(int)

# --- Explicit severe sepsis / septic shock (5-digit) ---
explicit_codes = ['99592','78552']
diagnoses["explicit_sepsis"] = diagnoses["ICD9_CODE"].str[:5].isin(explicit_codes).astype(int)

## 4. Mechanical ventilation (from PROCEDURES_ICD)
Procedure codes 9670, 9671, 9672 — note these come from the **procedures** table, not diagnoses.

In [ ]:
mech_vent_codes = ['9670', '9671', '9672']
procedures["mech_vent"] = procedures["ICD9_CODE"].isin(mech_vent_codes).astype(int)

## 5. Aggregate to HADM_ID level and apply the Angus rule

In [ ]:
# Which admissions have at least one code of each type
infection_hadm  = set(diagnoses.loc[diagnoses["infection"] == 1, "HADM_ID"])
organ_hadm      = set(diagnoses.loc[diagnoses["organ_dysfunction"] == 1, "HADM_ID"])
explicit_hadm   = set(diagnoses.loc[diagnoses["explicit_sepsis"] == 1, "HADM_ID"])
mechvent_hadm   = set(procedures.loc[procedures["mech_vent"] == 1, "HADM_ID"])

# Build a per-admission flag table starting from ALL admissions
angus = admissions[["SUBJECT_ID", "HADM_ID"]].copy()
angus["infection"]         = angus["HADM_ID"].isin(infection_hadm).astype(int)
angus["organ_dysfunction"] = angus["HADM_ID"].isin(organ_hadm).astype(int)
angus["explicit_sepsis"]   = angus["HADM_ID"].isin(explicit_hadm).astype(int)
angus["mech_vent"]         = angus["HADM_ID"].isin(mechvent_hadm).astype(int)

# Final Angus flag (exactly the SQL's CASE logic)
angus["angus"] = (
    (angus["explicit_sepsis"] == 1)
    | ((angus["infection"] == 1) & (angus["organ_dysfunction"] == 1))
    | ((angus["infection"] == 1) & (angus["mech_vent"] == 1))
).astype(int)

angus["angus"].value_counts()

,count
angus,
0,43722
1,15254


In [ ]:
# Sanity check: how many Angus-positive admissions?
print("Total admissions:", angus["HADM_ID"].nunique())
print("Angus-positive admissions:", angus.loc[angus["angus"]==1, "HADM_ID"].nunique())
print("Angus prevalence: {:.2%}".format(angus["angus"].mean()))

Total admissions: 58976
Angus-positive admissions: 15254
Angus prevalence: 25.86%


## 6. AKI label (comparison phenotype, ICD-9 584.x)

In [ ]:
aki_codes = ["5845", "5846", "5847", "5848", "5849"]
aki_hadm = set(diagnoses.loc[diagnoses["ICD9_CODE"].isin(aki_codes), "HADM_ID"])

## 7. Attach labels to ICU stays and build the cohort

In [ ]:
cohort = icustays.copy()

# Map the Angus flag onto ICU stays via HADM_ID
angus_map = angus.set_index("HADM_ID")["angus"]
cohort["Sepsis_Angus"] = cohort["HADM_ID"].map(angus_map).fillna(0).astype(int)
cohort["AKI"] = cohort["HADM_ID"].isin(aki_hadm).astype(int)

cohort.head()

In [ ]:
print(cohort["Sepsis_Angus"].value_counts())
print(cohort["AKI"].value_counts())
pd.crosstab(cohort["Sepsis_Angus"], cohort["AKI"])

Sepsis_Angus
0    23436
1    10124
Name: count, dtype: int64
AKI
0    26213
1     7347
Name: count, dtype: int64


AKI,0,1
Sepsis_Angus,,
0,21049,2387
1,5164,4960


## 8. Merge demographics (PATIENTS + ADMISSIONS)

In [ ]:
cohort = cohort.merge(
    patients[["SUBJECT_ID", "GENDER", "DOB"]],
    on="SUBJECT_ID", how="left"
)

cohort = cohort.merge(
    admissions[[
        "HADM_ID", "ADMITTIME", "DISCHTIME", "DEATHTIME",
        "ADMISSION_TYPE", "ETHNICITY", "HOSPITAL_EXPIRE_FLAG", "HAS_CHARTEVENTS_DATA"
    ]],
    on="HADM_ID", how="left"
)

## 9. Age (handle MIMIC date-shift overflow + 90+ capping)

In [ ]:
cohort["DOB"]      = pd.to_datetime(cohort["DOB"], errors="coerce")
cohort["ADMITTIME"] = pd.to_datetime(cohort["ADMITTIME"], errors="coerce")
cohort["INTIME"]    = pd.to_datetime(cohort["INTIME"], errors="coerce")
cohort["OUTTIME"]   = pd.to_datetime(cohort["OUTTIME"], errors="coerce")

# Year-difference avoids timedelta overflow from the shifted future dates
cohort["AGE"] = cohort["ADMITTIME"].dt.year - cohort["DOB"].dt.year
before_birthday = (
    (cohort["ADMITTIME"].dt.month < cohort["DOB"].dt.month)
    | ((cohort["ADMITTIME"].dt.month == cohort["DOB"].dt.month)
       & (cohort["ADMITTIME"].dt.day < cohort["DOB"].dt.day))
)
cohort.loc[before_birthday, "AGE"] -= 1

# MIMIC shifts ages >89 to ~300; cap at 90 per MIMIC convention
cohort.loc[cohort["AGE"] > 89, "AGE"] = 90
cohort["AGE"].describe()

,AGE
count,61532.000000
mean,55.161688
std,26.817771
min,0.000000
25%,44.000000
50%,62.000000
75%,76.000000
max,90.000000


## 10. Inclusion / exclusion + first ICU stay per patient

In [ ]:
cohort = cohort[(cohort["AGE"] >= 18) & (cohort["AGE"] <= 90)].copy()   # adults, valid age
cohort = cohort[cohort["LOS"] >= 1].copy()       # ICU LOS >= 1 day

# keep first ICU stay per patient
cohort = (cohort.sort_values("INTIME")
                .drop_duplicates(subset="SUBJECT_ID", keep="first")
                .copy())

print("Final cohort ICU stays:", cohort.shape[0])
print(cohort["Sepsis_Angus"].value_counts())
print(cohort["AKI"].value_counts())

Final cohort ICU stays: 33560
Sepsis_Angus
0    23436
1    10124
Name: count, dtype: int64
AKI
0    26213
1     7347
Name: count, dtype: int64


In [ ]:
cohort.to_csv(data_path("final_cohort_angus.csv"), index=False)
print("saved:", data_path("final_cohort_angus.csv"))

## 11. Descriptive statistics (for Section 3.2 / 4.1)

In [ ]:
print("Age:")
print(cohort["AGE"].describe())
print("\nMedian age:", cohort["AGE"].median())
print("\nGender %:")
print(cohort["GENDER"].value_counts(normalize=True) * 100)
print("\nLOS median:", cohort["LOS"].median())

Age:
count    33560.000000
mean        63.678963
std         17.232139
min         18.000000
25%         53.000000
50%         66.000000
75%         77.000000
max         90.000000
Name: AGE, dtype: float64

Median age: 66.0

Gender %:
GENDER
M    56.772944
F    43.227056
Name: proportion, dtype: float64

LOS median: 2.526


In [ ]:
# Phenotype breakdowns for the dissertation tables
print(cohort.groupby("Sepsis_Angus")["AGE"].describe())
print(cohort.groupby("AKI")["AGE"].describe())
print(pd.crosstab(cohort["Sepsis_Angus"], cohort["FIRST_CAREUNIT"]))
print(pd.crosstab(cohort["AKI"], cohort["FIRST_CAREUNIT"]))

                count       mean        std   min   25%   50%   75%   max
Sepsis_Angus                                                             
0             23436.0  62.673878  17.382034  18.0  51.0  64.0  77.0  90.0
1             10124.0  66.005630  16.649678  18.0  55.0  68.0  79.0  90.0
       count       mean        std   min   25%   50%   75%   max
AKI                                                             
0    26213.0  62.594858  17.392294  18.0  51.0  64.0  77.0  90.0
1     7347.0  67.546890  16.063737  18.0  57.0  70.0  80.0  90.0
FIRST_CAREUNIT   CCU  CSRU  MICU  SICU  TSICU
Sepsis_Angus                                 
0               3588  6238  6416  3966   3228
1               1301   874  5272  1546   1131
FIRST_CAREUNIT   CCU  CSRU  MICU  SICU  TSICU
AKI                                          
0               3583  6435  7590  4680   3925
1               1306   677  4098   832    434
